# 📘 Seismic Data Processing Notebook
FieldCamp II - June 2025

This notebook guides you through the steps of reading, visualizing, and processing seismic shot data using Python. It is designed for students new to Python and seismology.

---

In [ ]:
# 🧰 Step 1: Import necessary libraries
import pandas as pd        # for reading and working with tabular data
import os                  # for working with file paths
import numpy as np         # for numerical operations
import matplotlib.pyplot as plt  # for plotting
from obspy import UTCDateTime    # to work with seismic time formats

# Import helper functions from our custom `seismic` module
from seismic import (
    separate_shot_groups,       # separates shot data by group (e.g., P, S waves)
    plot_shot_data,             # plots summary data for shots
    plot_shot_geometry,         # plots geometry of sources and receivers
    read_waveforms,             # reads seismic waveform data from disk
    get_receiver_data,          # formats and cleans receiver data
    get_shots_data,             # merges and cleans shot location and time data
    process_and_export_shots    # performs waveform slicing, filtering, plotting, exporting
)

In [ ]:
# 📁 Step 2: Load input data (CSV files)

receiver_geometry_path = "/groups/igonin/ecastillo/FieldCampII_2025/data/seismic/June_14/receiver_geometry.csv"
shots_labeled_path = "/groups/igonin/ecastillo/FieldCampII_2025/data/seismic/June_14/shots_labeled.csv"
source_geometry_path = "/groups/igonin/ecastillo/FieldCampII_2025/data/seismic/June_14/source_geometry.csv"

out_folder = "/groups/igonin/ecastillo/FieldCampII_2025/out"

shots_labeled = pd.read_csv(shots_labeled_path)
source_geometry = pd.read_csv(source_geometry_path)
receiver_geometry = pd.read_csv(receiver_geometry_path)

receiver_geometry = get_receiver_data(receiver_geometry)
shots_geometry = get_shots_data(shots_labeled, source_geometry)
shots_groups = separate_shot_groups(shots_geom=shots_geometry)

In [ ]:
# 🗺️ Step 3: Visualize shot geometry and data

for group_name, shots in shots_groups.items():
    print(f"Group: {group_name}, Number of subgroups: {len(shots.groupby('shot_group'))}")
    print(f"Shots per subgroup: {shots.groupby('shot_group').size().to_dict()}")

    plot_shot_data(
        shots,
        save_path=os.path.join(out_folder, f"shots_{group_name}_data.png")
    )

    plot_shot_geometry(
        shots,
        receiver=receiver_geometry,
        save_path=os.path.join(out_folder, f"shots_{group_name}_geometry.png")
    )

In [ ]:
# 📦 Step 4: Download and extract seismic data
!wget -O smart_solo_data.zip "https://www.dropbox.com/scl/fo/tkeerdr54zz6r3psn61y1/ABIXTjSj-xDSgWm9sH-jjks?rlkey=jyw3s1cfk8mm3tcu5vuy8u8rk&e=1&dl=1"
!unzip smart_solo_data.zip -d smart_solo_data

In [ ]:
# Define waveform folder and time window
smart_solo_folder = "/groups/igonin/ecastillo/FieldCampII_2025/data_bck/smart_solo_data"
starttime = UTCDateTime(shots_groups["P"]["time"].min()) - 5
endtime = UTCDateTime(shots_groups["P"]["time"].max()) + 10

st = read_waveforms(
    folder_path=smart_solo_folder,
    component="Z",
    starttime=starttime,
    endtime=endtime
)

In [ ]:
# ⚙️ Step 5: Define processing parameters
processing = {
    "phase": "P",
    "left_seconds": 0.001,
    "right_seconds": 0.6,
    "delay_dict": {
        "2": 0.08,
        "37": 0.2
    },
    "apply_filter": True,
    "freqmin": 20,
    "freqmax": 60,
    "normalization": True,
    "plot": True,
    "only_specific_shots": [],
    "export_segy": False,
    "verbose": True
}

In [ ]:
# 🧪 Step 6: Process seismic shots
process_and_export_shots(
    st=st,
    shots_groups=shots_groups,
    receiver_geometry=receiver_geometry,
    out_folder=out_folder,
    phase=processing["phase"],
    left_seconds=processing["left_seconds"],
    right_seconds=processing["right_seconds"],
    delay_dict=processing["delay_dict"],
    apply_filter=processing["apply_filter"],
    freqmin=processing["freqmin"],
    freqmax=processing["freqmax"],
    normalization=processing["normalization"],
    plot=processing["plot"],
    only_specific_shots=[1],
    export_segy=processing["export_segy"],
    verbose=processing["verbose"]
)